## Decorator

---

> **In one line.** A decorator is a map $D : Y^X \to Y^X$ that takes a function and returns a *same-shaped* function with extra behavior wrapped around it. You describe it by one rule $h(f, x)$, and decorators stack because they compose.

### 1. The space we work in

Let $X$ and $Y$ be sets ($X$ = inputs, $Y$ = outputs). The **function space** is the set of all functions from $X$ to $Y$:

$$Y^X \;:=\; \{\, f \mid f : X \to Y \,\}.$$

A single function $f$ is one **point** of $Y^X$. The thing we wrap is such a point: $f \in Y^X$.

> *Aside.* The power notation is literal: for finite sets there are $|Y|^{|X|}$ functions, hence "$Y$ to the $X$." In category language $Y^X$ is the **exponential object in $\mathbf{Set}$**.

### 2. What a decorator is

$$\boxed{\,D : Y^X \to Y^X\,}\qquad D \in \operatorname{End}\!\big(Y^X\big).$$

It is an **endo-operator**: *same space in, same space out.* It eats a function and returns another function of the **same** type. $\operatorname{End}(Y^X)$ is the set of all such maps.

### 3. How you describe one: the kernel $h$

To pin down a specific $D$, give one rule for what the wrapped function does when called. That rule is the **augmentation kernel** $h$ (augmentation = the added behavior). It takes the original $f$ **and** an input $x$, and returns the output:

$$h : Y^X \times X \longrightarrow Y, \qquad D(f) := \big(\, x \mapsto h(f, x) \,\big).$$

So $h$ is the **body** of the wrapped function, with both $f$ and $x$ in scope.

### 4. Why $D$ and $h$ are the same data (currying)

$D$ takes $f$ and returns a function; $h$ takes $(f, x)$ and returns an output. **Currying** says these carry identical information:

- **$h \to D$:** fix $f$, leave $x$ open; the leftover $x \mapsto h(f,x)$ *is* $D(f)$.
- **$D \to h$:** apply $D$ to $f$, then apply the result to $x$; the output is $h(f,x)$.

$$\underbrace{\operatorname{End}\!\big(Y^X\big)}_{\text{all } D} \;=\; \big(Y^X\big)^{Y^X} \;\cong\; \underbrace{Y^{\,(Y^X \times X)}}_{\text{all } h}, \qquad\qquad D(f)(x) = h(f, x).$$

**Takeaway:** you only ever write the kernel $h$; the decorator $D$ comes for free.

Because $h$ *receives* $f$ (rather than being forced to run it first), it may call $f$ **zero** times (cache hit), **once** (usual), or **many** times (retry). That is what makes a decorator more than post-composition $a \circ f$, which always runs $f$ once.

### 5. Why they stack: the monoid $\operatorname{End}(Y^X)$

A **monoid** is a set with an associative combine and a do-nothing element. Endo-operators give one under composition:

$$\big(\operatorname{End}(Y^X),\ \circ,\ \operatorname{id}_{Y^X}\big), \qquad (D_3 \circ D_2 \circ D_1)(f) = D_3\big(D_2(D_1(f))\big).$$

The unit $\operatorname{id}_{Y^X}$ (with $\operatorname{id}(f) = f$) is the **trivial decorator**, adding nothing.

### 6. The three conditions

| Condition | Statement |
|---|---|
| **Type preservation** | $D$ is endo, so $D(f) \in Y^X$ automatically: same $X$, same $Y$. |
| **Non-modification** | $f$ is untouched; $D(f)$ is a *new* element built around it. |
| **Composability** | exactly the monoid structure: stacking is closed, associative, unital. |

### 7. The common shape: before / after hooks

When $h$ calls $f$ **exactly once**, it splits into a *pre* map $b : X \to X$ and a *post* map $a : Y \to Y$:

$$h(f, x) = a\big(f(b(x))\big), \qquad D(f) = a \circ f \circ b.$$

| Kernel does | $D(f)$ | Example |
|---|---|---|
| post-process only | $a \circ f$ &nbsp;($b = \operatorname{id}_X$) | "double the result" |
| pre-process only | $f \circ b$ &nbsp;($a = \operatorname{id}_Y$) | validate / log the input |
| skip $f$ sometimes | does not factor; needs full $h(f,x)$ | caching, auth guard, retry |

The narrow form $D(f) = a \circ f$ is only the first row; keeping $f$ inside $h$ is what buys the third.

> *Aside (many methods).* A real object has several methods $g_i : X_i \to Y_i$. Bundle them into a product $I = \prod_i Y_i^{X_i}$; a decorator is then an endo-operator $D : I \to I$, the theory above applied per coordinate.


### Exercise 1 — Logging Decorator

---

**Scenario:** A `TextEditor` has `write(text)`. You want to log every call **without touching `TextEditor`**. The decorator wraps it, adding logging as the augmentation $h$.

**Your task:** Write a `LoggingDecorator` wrapping any text editor. It logs **before** every `write()` call, then delegates to $f$.

```python
editor = LoggingDecorator(TextEditor())   # D(f)
editor.write("Hello")
# [LOG] write() called with: Hello        <- h (the added behavior)
# Hello                                    <- f(x) (the original)
```

**Hints**

- The decorator holds `self._editor = editor` — this stores $f$. Its `write()` is the decorated call: print the log ($h$), **then** call `self._editor.write(text)` ($f$). Since the log runs *before* $f$, this is the **pre-hook** shape $D(f)(x) = f(b(x))$ where the "$b$" step is the logging side-effect.
- The decorator must expose the **same** interface as `TextEditor` — same method name `write`. This is the **type preservation** condition: $D(f) : X \rightarrow Y$, so the client can't tell it's talking to a decorator.


In [5]:
#--------------------------------
# Original (f) — you cannot change this

class TextEditor:
    def write(self, text):
        print(text)

#--------------------------------
# Decorator (D) — your task: log before delegating to f
import logging
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

class LoggingDecorator:
    def __init__(self, editor):
        self._editor = editor           # stores f

    def write(self, text):              # same interface as TextEditor (type preservation)
        # 1) log the call
        # 2) self._editor.write(text)                (f(x))
        logger.info(f"[LOG] write() called with: {text}")
        return self._editor.write(text)
        

#--------------------------------
editor = LoggingDecorator(TextEditor())   # D(f)
editor.write("Hello")
# expected:
# write() called with: Hello
# Hello


INFO:__main__:[LOG] write() called with: Hello


Hello


### Exercise 2 — Stacked Decorators (Coffee Shop)

---

**Scenario:** A `Coffee` has `cost()` and `description()`. Add-ons (`Milk`, `Sugar`, `Vanilla`) each increment the cost and extend the description — and can be **stacked in any order**.

**Your task:** Build add-on decorators that compose freely: $D_{\text{Vanilla}}\big(D_{\text{Milk}}(D_{\text{Sugar}}(f))\big)$.

```python
drink = Vanilla(Milk(Sugar(Coffee())))
print(drink.cost())         # sum of all layers
print(drink.description())  # Coffee, Sugar, Milk, Vanilla
```

**Hints**

- Each add-on stores the inner drink: `self._drink = drink`. Then `cost()` returns `self._drink.cost() + self.added_cost` — this is $h(f,x)$ where $h$ adds the increment ($f$ = the inner drink, untouched).
- Swap the wrapping order: the **total cost stays the same**, only the **description order changes**. This confirms the **composability** condition $D_3(D_2(D_1(f)))$.


In [ ]:
#--------------------------------
# Original (f) — the base drink, you cannot change this

class Coffee:
    def cost(self):
        return 2.0

    def description(self):
        return "Coffee"

#--------------------------------
# Add-on decorators (D) — each wraps a drink and adds to cost + description
# Every add-on exposes the SAME interface: cost() and description()  (type preservation)

class Sugar:
    added_cost = 0.25
    def __init__(self, drink):
        self._drink = drink                         # stores f (inner drink)
    def cost(self):
        return self._drink.cost() + self.added_cost # self._drink.cost() + self.added_cost
                                         
    def description(self):
        return self._drink.description() + ", Sugar" # self._drink.description() + ", Sugar"

class Milk:
    added_cost = 0.50
    def __init__(self, drink):
        self._drink = drink
    def cost(self):
        return self._drink.cost() + self.added_cost

    def description(self):
        return self._drink.description() + ", Milk"

class Vanilla:
    added_cost = 0.75
    def __init__(self, drink):
        self._drink = drink
    def cost(self):
        return self._drink.cost() + self.added_cost

    def description(self):
        return self._drink.description() + ", Vanilla"

#--------------------------------
drink = Vanilla(Milk(Sugar(Coffee())))              # D_Vanilla(D_Milk(D_Sugar(f)))
print(drink.cost())                                 # expected: 3.5
print(drink.description())                          # expected: Coffee, Sugar, Milk, Vanilla

# composability check — different order, SAME cost, different description order:
other = Sugar(Vanilla(Milk(Coffee())))
print(other.cost())                                 # expected: 3.5  (unchanged)
print(other.description())                          # expected: Coffee, Milk, Vanilla, Sugar


3.5
Coffee, Sugar, Milk, Vanilla
3.5
Coffee, Milk, Vanilla, Sugar


### Exercise 3 — Caching Decorator (the kernel that skips $f$)

---

**Scenario:** A `SlowDatabase` has `query(key)` that is expensive to run. You want to **cache** results: the first call for a key runs the real query, but repeat calls for the same key return the stored answer **without calling $f$ again**.

This is the case the narrow form $D(f) = a \circ f$ **cannot** express, because $a \circ f$ always runs $f$. Caching needs the full kernel $h(f, x)$, where $h$ decides whether to call $f$ at all.

**Your task:** Write a `CachingDecorator` that wraps any object with a `query(key)` method and remembers results.

```python
db = CachingDecorator(SlowDatabase())
db.query("users")   # MISS -> runs the real query
db.query("users")   # HIT  -> returns cached value, f NOT called
```

**Hints**

- Keep a dict `self._cache = {}`. In `query(key)`: if `key` is already in the cache, return it and **do not** call $f$ (the kernel applies $f$ **zero** times). Otherwise call `self._db.query(key)`, store it, and return it (apply $f$ **once**).
- Same interface as `SlowDatabase` (method `query`), so **type preservation** holds: the caller can't tell it's cached.
- Print a `MISS`/`HIT` line so you can *see* when $f$ is and isn't called.


In [2]:
# --------------------------------
# Original (f) — expensive, you cannot change this

class SlowDatabase:
    def query(self, key):
        print(f"[DB] running expensive query for: {key}")   # proof f actually ran
        return f"result({key})"

# --------------------------------
# Decorator (D) — your task: cache results, skip f on a hit

class CachingDecorator:
    def __init__(self, db):
        self._db = db                       # stores f
        self._cache = {}                    # remembered results

    def query(self, key):                   # same interface as SlowDatabase (type preservation)
        # if key in self._cache: HIT  -> return cached, do NOT call f  (h applies f 0 times)
        if key in self._cache:
            print(f"[CACHE] HIT: {key}")
            return self._cache[key]
        else:
            # else:                  MISS -> call self._db.query(key), store, return  (f once)
            print(f"[CACHE] MISS: {key}")
            result = self._db.query(key)
            # Add the result to the cache
            self._cache[key] = result
            return result

# --------------------------------
db = CachingDecorator(SlowDatabase())
print(db.query("users"))    # MISS: runs the real query
print("\n--------\n")
print(db.query("users"))    # HIT : cached, no [DB] line
print("\n--------\n")
print(db.query("orders"))   # MISS: new key
print("\n--------\n")
# expected:
# MISS users
# [DB] running expensive query for: users
# result(users)
# HIT users
# result(users)
# MISS orders
# [DB] running expensive query for: orders
# result(orders)

[CACHE] MISS: users
[DB] running expensive query for: users
result(users)

--------

[CACHE] HIT: users
result(users)

--------

[CACHE] MISS: orders
[DB] running expensive query for: orders
result(orders)

--------

